#### 1. Idioma geral, instalação das bibiotecas necessárias e teste do microfone:

In [ ]:
pip install pipwin


In [ ]:
# Determina a linguagem que será usada pelo usuário
language = "pt-BR"

In [ ]:
# Instala as bibliotecas necessárias para o programa
%pip install python-dotenv google-generativeai SpeechRecognition pyaudio gTTS

#### 2. Colocar o texto para o chat resumir e transformar em audio

In [ ]:
import speech_recognition as sr
from google import genai
from gtts import gTTS
import IPython.display as ipd
import os
from dotenv import load_dotenv


# 1. Carrega as variáveis do arquivo .env
load_dotenv()

# 2. Busca a chave de forma segura (Certifique-se que no seu .env está GOOGLE_API_KEY=suachave)
minha_chave = os.getenv("GOOGLE_API_KEY=sua_chave_aqui")

# 3. Configura o Cliente da API (Versão mais atual da biblioteca)
client = genai.Client(api_key=minha_chave)

prompt_sistema = (
    "Você é um resumidor de textos para áudio. "
    "Responda SEM Markdown, SEM emojis e escreva números por extenso. "
    "Crie um resumo curto com 3 pontos principais do texto fornecido."
)

# Entrada do Texto Longo (Manual)
print("--- 1. PREPARAÇÃO ---")
textao_para_resumir = input("Cole o texto longo aqui e dê Enter: ")

r = sr.Recognizer()

print("\n--- 2. COMANDO DE VOZ ---")
print("Diga 'RESUMIR' ou 'ENVIAR' para eu processar o texto...")

while True:
    try:
        with sr.Microphone() as source:
            print("Ouvindo...")
            r.adjust_for_ambient_noise(source, duration=0.8)
            audio = r.listen(source)
            comando_voz = r.recognize_google(audio, language="pt-BR").lower()
            
            print(f"Comando detectado: {comando_voz}")

            if "enviar" in comando_voz or "resumir" in comando_voz:
                print("Processando com Gemini...")

                # Chamada corrigida para a estrutura da biblioteca 'google-genai'
                response = client.models.generate_content(
                    model="gemini-2.5-flash", 
                    config={'system_instruction': prompt_sistema},
                    contents=textao_para_resumir 
                )

                response_audio_text = response.text
                print(f"\nResumo do Gemini:\n{response_audio_text}")

                # Gerar e tocar áudio
                tts = gTTS(text=response_audio_text, lang='pt-br')
                tts.save("resposta.mp3")
                display(ipd.Audio("resposta.mp3", autoplay=True))
                break
                
            elif "parar" in comando_voz or "sair" in comando_voz:
                print("Saindo...")
                break

    except sr.UnknownValueError:
        continue 
    except Exception as e:
        print(f"Ocorreu um erro: {e}")
        break